# Expense Extraction Dev Notebook

Use this notebook to test the same pipeline the app uses, one step at a time:

1. Pick one or more files from `uploads/`
2. Extract text from each file
3. Combine multiple PDFs/text files into one vLLM batch payload
4. Inspect the generated prompt
5. Send the batch to vLLM
6. Inspect the parsed JSON output


In [1]:
from pathlib import Path
from pprint import pprint

from app.config import settings
from app.vllm_client import build_prompt, generate_expenses_from_text
from app.parser import encode_image_to_base64, extract_text_from_file

UPLOADS_DIR = settings.storage_dir
BATCHABLE_SUFFIXES = {".pdf", ".txt", ".csv"}

print(f"vLLM URL: {settings.vllm_url}")
print(f"Model: {settings.vllm_model}")
print(f"Uploads dir: {UPLOADS_DIR.resolve()}")


vLLM URL: http://host.docker.internal:8000
Model: Qwen/Qwen3.6-35B-A3B-FP8
Uploads dir: /home/nvidia/Desktop/ExpenseTracker/uploads


In [2]:
available_files = sorted(
    path
    for path in UPLOADS_DIR.glob("*")
    if path.suffix.lower() in BATCHABLE_SUFFIXES | {".png", ".jpg", ".jpeg", ".webp", ".heic"}
)

[path.name for path in available_files[:50]]


[]

Edit `selected_files` below if you want to test a specific set. If you leave it empty, the notebook will default to the first available file.

In [ ]:
selected_files = [
    # UPLOADS_DIR / "statement_a.pdf",
    # UPLOADS_DIR / "statement_b.pdf",
]

if not selected_files and available_files:
    selected_files = [available_files[0]]

if not selected_files:
    raise FileNotFoundError("No files found in uploads/. Add a file or set selected_files manually.")

selected_files


In [ ]:
def load_document(path: Path) -> dict:
    text = extract_text_from_file(path) or ""
    image_b64 = encode_image_to_base64(path)
    return {
        "path": path,
        "name": path.name,
        "suffix": path.suffix.lower(),
        "text": text,
        "text_chars": len(text),
        "is_batchable": path.suffix.lower() in BATCHABLE_SUFFIXES,
        "has_image_payload": bool(image_b64),
        "image_b64": image_b64,
    }


documents = [load_document(path) for path in selected_files]

[
    {
        "name": doc["name"],
        "suffix": doc["suffix"],
        "text_chars": doc["text_chars"],
        "is_batchable": doc["is_batchable"],
        "has_image_payload": doc["has_image_payload"],
    }
    for doc in documents
]


In [ ]:
for doc in documents:
    print("=" * 100)
    print(doc["name"])
    print(doc["text"][:1500] if doc["text"] else "<no extracted text>")


In [ ]:
def combine_document_text(documents: list[dict]) -> str:
    sections = []
    for index, doc in enumerate(documents, start=1):
        if not doc["text"]:
            continue
        sections.append(
            "\n".join(
                [
                    f"--- BEGIN DOCUMENT {index}: {doc['name']} ---",
                    doc["text"],
                    f"--- END DOCUMENT {index}: {doc['name']} ---",
                ]
            )
        )
    return "\n\n".join(sections)


combined_text = combine_document_text(documents)
image_data_uris = [doc["image_b64"] for doc in documents if doc["image_b64"]] if not combined_text else None

print(f"Selected files: {len(documents)}")
print(f"Combined text chars: {len(combined_text)}")
print(f"Using image payload: {bool(image_data_uris)}")


In [ ]:
if combined_text:
    print(combined_text[:5000])
else:
    print("<combined text is empty>")


In [ ]:
prompt_text = build_prompt(combined_text or "Use the attached image.")
print(prompt_text[:5000])


In [ ]:
raw_response, parsed_response = generate_expenses_from_text(
    combined_text or "Use the attached images.",
    image_data_uris=image_data_uris,
)

print(raw_response[:5000])


In [ ]:
pprint(parsed_response)


In [ ]:
expenses = (parsed_response or {}).get("expenses", [])
expenses
